# MCMC CMB Only (Paper I Dataset Separation)**Author**: Ricardo Alvim**Purpose**: Constrain Evaporating Universe using CMB distance priors---## Runtime: ~2-3 hours on Colab ProNote: This uses CMB distance priors (compressed likelihood), not full CLASS.

In [ ]:
# ============================================================
# INSTALLATION (run this cell first!)
# ============================================================
# Core packages are pre-installed in Colab
# !pip install numpy matplotlib scipy  # Already in Colab
print('Colab environment ready!')

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom scipy.integrate import quadimport emceefrom multiprocessing import Pool, cpu_countimport cornerimport jsonfrom datetime import datetimeimport warningswarnings.filterwarnings('ignore')print("="*70)print("MCMC CMB ONLY - Evaporating Universe")print("Using CMB distance priors (compressed likelihood)")print("="*70)n_cores = min(cpu_count(), 32)print(f'Available CPU cores: {n_cores}')

In [ ]:
# =============================================================# CMB DISTANCE PRIORS (Planck 2018)# =============================================================# R = sqrt(Omega_m * H0^2) * D_A(z*) / c# l_a = pi * D_A(z*) / r_s(z*)# z* = 1089.92 (recombination redshift)z_star = 1089.92# Planck 2018 measurementsR_obs = 1.7502  # +/- 0.0046R_err = 0.0046la_obs = 301.471  # +/- 0.090la_err = 0.090omega_b_obs = 0.02236  # omega_b = Omega_b * h^2omega_b_err = 0.00015# Correlation matrixrho_R_la = 0.46print(f"CMB priors: R = {R_obs} +/- {R_err}, l_a = {la_obs} +/- {la_err}")

In [ ]:
# =============================================================# EVAPORATING UNIVERSE MODEL# =============================================================c = 299792.458  # km/sdef w_de(z, w0, z_trans):if z_trans <= 0.01:return -1.0if z > z_trans:return -1.0delta_w = w0 - (-1.0)return -1.0 + delta_w * (1 - z/z_trans)**2def E_z(z, Omega_m, w0, z_trans):Omega_de = 1 - Omega_mw = w_de(z, w0, z_trans)rho_de = Omega_de * (1 + z)**(3*(1+w))return np.sqrt(Omega_m * (1+z)**3 + rho_de)def comoving_distance(z, H0, Omega_m, w0, z_trans):def integrand(zp):return 1.0 / E_z(zp, Omega_m, w0, z_trans)result, _ = quad(integrand, 0, z, limit=500)return c / H0 * resultdef angular_diameter_distance(z, H0, Omega_m, w0, z_trans):DM = comoving_distance(z, H0, Omega_m, w0, z_trans)return DM / (1 + z)def sound_horizon(omega_b, Omega_m, h):"""Approximate sound horizon at recombination."""omega_m = Omega_m * h**2# Fitting formula from Eisenstein & Hu 1998rs = 44.5 * np.log(9.83 / omega_m) / np.sqrt(1 + 10 * omega_b**0.75)return rs  # Mpcprint("Model defined.")

In [ ]:
#=============================================================#CMBOBSERVABLES(Correctedformulas)#=============================================================#Reference:Chen,Kumar&Ratra2017(arXiv:1606.04684)#Planck2018distancepriors:arXiv:1807.06209defcalc_R(H0,Omega_m,w0,z_trans):"""ShiftparameterR=sqrt(Omega_m)*D_M(z*)/D_HwhereD_H=c/H0,D_M=comovingdistance"""D_M=comoving_distance(z_star,H0,Omega_m,w0,z_trans)#inMpcD_H=c/H0#inMpcR=np.sqrt(Omega_m)*D_M/D_HreturnRdefcalc_la(H0,Omega_m,w0,z_trans,omega_b):"""Acousticscalel_a=pi*(1+z*)*D_A(z*)/r_s(z*)UsingPlanck2018soundhorizonfittingformula."""DA=angular_diameter_distance(z_star,H0,Omega_m,w0,z_trans)#inMpch=H0/100omega_m=Omega_m*h**2#SoundhorizonfromPlanck2018fittingformula(Eq.6ofChenetal.)#r_s=55.154*exp[-72.3*(omega_nu+0.0006)^2]/(omega_m^0.25351*omega_b^0.12807)#Simplifiedforomega_nu=0:omega_nu=0.0006r_s=55.154*np.exp(-72.3*(omega_nu+0.0006)**2)/(omega_m**0.25351*omega_b**0.12807)#l_a=pi*DA*(1+z*)/r_s[bothinMpc]la=np.pi*DA*(1+z_star)/r_sreturnla#VerifywithPlanckbest-fitR_test=calc_R(67.36,0.3153,-1.0,0.22)la_test=calc_la(67.36,0.3153,-1.0,0.22,0.02237)print(f"CMBobservablesdefined(corrected).")print(f"TestwithPlanckparams:R={R_test:.4f}(obs:1.7502),l_a={la_test:.2f}(obs:301.47)")

In [ ]:
#=============================================================#LIKELIHOODWITHCORRELATION#=============================================================deflog_likelihood(theta):H0,Omega_m,w0,z_trans,omega_b=theta#CalculateCMBobservablestry:R_pred=calc_R(H0,Omega_m,w0,z_trans)la_pred=calc_la(H0,Omega_m,w0,z_trans,omega_b)except:return-np.inf#ResidualsdR=(R_pred-R_obs)/R_errdla=(la_pred-la_obs)/la_errd_omega_b=(omega_b-omega_b_obs)/omega_b_err#Chi-squaredwithcorrelationchi2_Rla=(dR**2+dla**2-2*rho_R_la*dR*dla)/(1-rho_R_la**2)chi2_omega=d_omega_b**2return-0.5*(chi2_Rla+chi2_omega)deflog_prior(theta):H0,Omega_m,w0,z_trans,omega_b=theta#FlatpriorscenteredonPlanckifnot(55<H0<85):return-np.infifnot(0.20<Omega_m<0.45):return-np.infifnot(-1.5<w0<-0.5):return-np.infifnot(0.05<z_trans<0.8):return-np.infifnot(0.020<omega_b<0.025):return-np.infreturn0.0deflog_probability(theta):lp=log_prior(theta)ifnotnp.isfinite(lp):return-np.infll=log_likelihood(theta)ifnotnp.isfinite(ll):return-np.infreturnlp+llprint("Likelihooddefinedwithcorrecteddistancepriors.")print("Expected:H0~67,Omega_m~0.31(Planck),w0unconstrained")

In [ ]:
#=============================================================#RUNMCMC#=============================================================initial=np.array([73.0,0.30,-1.15,0.22,0.0224])ndim=len(initial)nwalkers=32nsteps=5000pos=initial+1e-4*np.random.randn(nwalkers,ndim)print(f"RunningMCMC:{nwalkers}walkers,{nsteps}steps...")print("Thiswilltake~2-3hoursonColabPro")withPool(n_cores)aspool:sampler=emcee.EnsembleSampler(nwalkers,ndim,log_probability,pool=pool)sampler.run_mcmc(pos,nsteps,progress=True)print("MCMCcomplete!")

In [ ]:
# =============================================================# ANALYZE RESULTS# =============================================================burnin = 1000samples = sampler.get_chain(discard=burnin, flat=True)labels = [r'$H_0$', r'$\Omega_m$', r'$w_0$', r'$z_{trans}$', r'$\omega_b$']means = np.mean(samples, axis=0)stds = np.std(samples, axis=0)print("\n" + "="*50)print("CMB ONLY RESULTS")print("="*50)for i, (label, mean, std) in enumerate(zip(labels, means, stds)):print(f"{label}: {mean:.5f} +/- {std:.5f}")

In [ ]:
# =============================================================# CORNER PLOT# =============================================================fig = corner.corner(samples, labels=labels, quantiles=[0.16, 0.5, 0.84],show_titles=True, title_fmt='.4f')plt.suptitle('CMB Only Constraints', fontsize=14)plt.tight_layout()plt.savefig('mcmc_cmb_only_corner.png', dpi=150)plt.show()

In [ ]:
# =============================================================# SAVE RESULTS# =============================================================results = {"metadata": {"analysis": "MCMC CMB Only (distance priors)","date": datetime.now().isoformat(),"nwalkers": nwalkers,"nsteps": nsteps,"burnin": burnin},"parameters": {"H0": [float(means[0]), float(stds[0])],"Omega_m": [float(means[1]), float(stds[1])],"w0": [float(means[2]), float(stds[2])],"z_trans": [float(means[3]), float(stds[3])],"omega_b": [float(means[4]), float(stds[4])]},"maturity": "Paper Standard","figures": ["mcmc_cmb_only_corner.png"]}with open('mcmc_cmb_only_results.json', 'w') as f:json.dump(results, f, indent=2)np.save('mcmc_cmb_only_chain.npy', samples)print("Saved results!")try:from google.colab import filesfiles.download('mcmc_cmb_only_corner.png')files.download('mcmc_cmb_only_results.json')files.download('mcmc_cmb_only_chain.npy')except:print("Files saved locally.")